## Colab Runtime Setup
Run this cell first. It prepares the Colab environment, clones the repository if needed, installs dependencies, and checks GPU availability.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ENABLE_CLONE_FALLBACK = True
REPO_URL = "https://github.com/Hansen256/NLP_coursework_2.git"
COLAB_REPO_DIR = Path("/content/Coursework_2")

if IN_COLAB:
    if ENABLE_CLONE_FALLBACK and not COLAB_REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)])
    if COLAB_REPO_DIR.exists():
        os.chdir(COLAB_REPO_DIR)

    if Path("requirements.txt").exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "scikit-learn", "gensim", "nltk", "kagglehub", "kaggle"])

    gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
    gpu_name = gpu_name.splitlines()[0].strip() if gpu_name.strip() else ""
    print("Colab runtime ready at:", Path.cwd())
    if gpu_name and "NVIDIA-SMI" not in gpu_name:
        print("GPU detected:", gpu_name)
    else:
        print("GPU not detected. In Colab: Runtime -> Change runtime type -> GPU")

#  Google Drive Mount


In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import drive

    # Mounting Google Drive to save files during the session.
    drive.mount("/content/drive", force_remount=False)
    print("Google Drive mounted at /content/drive")
else:
    print("Skipping Google Drive mount: not running in Colab.")

## Colab: Kaggle Credentials + Dataset Download
Upload your Kaggle API key (`kaggle.json`), then download the dataset via `kagglehub` into `/content/data_raw` (Colab) so the rest of the notebook runs unchanged.

In [ ]:
# Upload kaggle.json, then this cell will install it for Kaggle/KaggleHub auth.
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import files

    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Please upload a file named kaggle.json")

    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("kaggle.json installed at /root/.kaggle/kaggle.json")
else:
    print("Skipping kaggle.json upload: not running in Colab.")

## Embedded Pipeline Modules (Standalone Mode)
This cell registers in-memory `src.*` modules so the notebook runs without uploading the `src` folder.

In [ ]:
import re
import sys
import types
from dataclasses import dataclass
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create src package and submodules in-memory so existing imports work without local src files.
src_pkg = sys.modules.get("src")
if src_pkg is None:
    src_pkg = types.ModuleType("src")
    src_pkg.__path__ = []
    sys.modules["src"] = src_pkg

data_loader_mod = types.ModuleType("src.data_loader")
preprocessing_mod = types.ModuleType("src.preprocessing")
representations_mod = types.ModuleType("src.representations")
similarity_mod = types.ModuleType("src.similarity")
evaluation_mod = types.ModuleType("src.evaluation")

COMMON_TEXT_COLUMN_NAMES = [
    "text",
    "tweet",
    "tweet_text",
    "content",
    "body",
    "message",
]

def _score_text_column(series: pd.Series) -> float:
    non_null = series.dropna().astype(str).str.strip()
    non_empty = non_null[non_null != ""]
    if non_empty.empty:
        return 0.0
    avg_length = float(non_empty.str.len().mean())
    fill_ratio = float(len(non_empty) / len(series))
    return (avg_length * 0.8) + (fill_ratio * 20.0)

def _detect_text_column(df: pd.DataFrame) -> str:
    lower_map = {col.lower(): col for col in df.columns}
    for candidate in COMMON_TEXT_COLUMN_NAMES:
        if candidate in lower_map:
            return lower_map[candidate]

    candidate_cols = []
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]):
            candidate_cols.append(col)

    if not candidate_cols:
        raise ValueError("No text-like column found in dataset.")

    best_col = max(candidate_cols, key=lambda col: _score_text_column(df[col]))
    return best_col

def load_tweets_dataframe(
    raw_dir: Path,
    minimum_rows: int = 1000,
    sample_size: int = 10000,
    random_state: int = 42,
):
    csv_files = sorted(raw_dir.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in raw data directory: {raw_dir}")

    selected_path = csv_files[0]
    df = pd.read_csv(selected_path)
    text_col = _detect_text_column(df)

    text_series = df[text_col].astype(str).str.strip()
    df = df[text_series != ""].copy()

    if len(df) < minimum_rows:
        raise ValueError(
            f"Dataset has {len(df)} rows after cleaning, below minimum required {minimum_rows}."
        )

    if sample_size > 0 and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=random_state).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    return df, text_col, selected_path

URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
HASHTAG_RE = re.compile(r"#\w+")
MENTION_RE = re.compile(r"@\w+")
REPEATED_CHARS_RE = re.compile(r"(.)\1{2,}")
EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE,
)
TOKEN_RE = re.compile(r"[a-z']+")

@dataclass
class NoiseStats:
    urls_count: int
    hashtags_count: int
    mentions_count: int
    emojis_count: int
    repeated_chars_count: int
    avg_chars_per_tweet: float

def _ensure_nltk_resources() -> None:
    resource_paths = {
        "corpora/stopwords": "stopwords",
        "corpora/wordnet": "wordnet",
        "corpora/omw-1.4": "omw-1.4",
    }
    for check_path, resource_name in resource_paths.items():
        try:
            nltk.data.find(check_path)
        except LookupError:
            nltk.download(resource_name, quiet=True)

def compute_noise_stats(text_series: pd.Series) -> NoiseStats:
    texts = text_series.fillna("").astype(str)
    urls_count = int(texts.str.count(URL_RE).sum())
    hashtags_count = int(texts.str.count(HASHTAG_RE).sum())
    mentions_count = int(texts.str.count(MENTION_RE).sum())
    emojis_count = int(texts.str.count(EMOJI_RE).sum())
    repeated_chars_count = int(texts.str.count(REPEATED_CHARS_RE).sum())
    avg_chars = float(texts.str.len().mean()) if len(texts) else 0.0

    return NoiseStats(
        urls_count=urls_count,
        hashtags_count=hashtags_count,
        mentions_count=mentions_count,
        emojis_count=emojis_count,
        repeated_chars_count=repeated_chars_count,
        avg_chars_per_tweet=avg_chars,
    )

def preprocess_text_series(text_series: pd.Series) -> pd.DataFrame:
    _ensure_nltk_resources()

    texts = text_series.fillna("").astype(str).str.lower()
    texts = texts.str.replace(URL_RE, " ", regex=True)
    texts = texts.str.replace(HASHTAG_RE, " ", regex=True)
    texts = texts.str.replace(MENTION_RE, " ", regex=True)
    texts = texts.str.replace(EMOJI_RE, " ", regex=True)
    texts = texts.str.replace(r"[^a-z\s']", " ", regex=True)
    texts = texts.str.replace(r"\s+", " ", regex=True).str.strip()

    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()

    clean_tokens = []
    clean_text = []
    for text in texts.tolist():
        tokens = [tok for tok in TOKEN_RE.findall(text) if tok and tok not in stop_words]
        lemmas = [lemmatizer.lemmatize(tok) for tok in tokens if len(tok) > 1]
        clean_tokens.append(lemmas)
        clean_text.append(" ".join(lemmas))

    return pd.DataFrame({"clean_text": clean_text, "clean_tokens": clean_tokens})

def build_bow(corpus, max_features: int = 20000, min_df: int = 2):
    vectorizer = CountVectorizer(max_features=max_features, min_df=min_df)
    matrix = vectorizer.fit_transform(corpus)
    return vectorizer, matrix

def build_tfidf(corpus, max_features: int = 20000, min_df: int = 2):
    vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df)
    matrix = vectorizer.fit_transform(corpus)
    return vectorizer, matrix

def corpus_vocabulary_size(tokenized_corpus) -> int:
    unique_tokens = {token for doc in tokenized_corpus for token in doc}
    return len(unique_tokens)

def top_terms_from_bow(vectorizer, matrix, top_k: int = 20):
    features = vectorizer.get_feature_names_out()
    scores = np.asarray(matrix.sum(axis=0)).ravel()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(features[i], float(scores[i])) for i in top_idx]

def top_terms_from_tfidf(vectorizer, matrix, top_k: int = 20):
    features = vectorizer.get_feature_names_out()
    scores = np.asarray(matrix.mean(axis=0)).ravel()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(features[i], float(scores[i])) for i in top_idx]

def train_word2vec(
    tokenized_corpus,
    vector_size: int = 100,
    window: int = 5,
    min_count: int = 5,
    workers: int = 1,
    epochs: int = 8,
    seed: int = 42,
):
    filtered_corpus = [doc for doc in tokenized_corpus if doc]
    if not filtered_corpus:
        raise ValueError("Tokenized corpus is empty after preprocessing.")

    model = Word2Vec(
        sentences=filtered_corpus,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        epochs=epochs,
        seed=seed,
    )
    return model

def most_similar_terms(w2v_model, query_terms, topn: int = 10):
    results = {}
    for term in query_terms:
        if term in w2v_model.wv:
            items = w2v_model.wv.most_similar(term, topn=topn)
            results[term] = [(word, float(score)) for word, score in items]
        else:
            results[term] = []
    return results

def rank_similarity_to_reference(
    tweets: pd.Series,
    reference_text: str,
    max_features: int = 20000,
    min_df: int = 2,
) -> pd.DataFrame:
    tweet_texts = tweets.fillna("").astype(str).tolist()
    docs = tweet_texts + [reference_text]

    vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df)
    tfidf = vectorizer.fit_transform(docs)

    tweet_vectors = tfidf[:-1]
    reference_vector = tfidf[-1]
    scores = cosine_similarity(tweet_vectors, reference_vector).ravel()

    ranked = pd.DataFrame(
        {
            "tweet": tweet_texts,
            "similarity_score": scores,
        }
    ).sort_values("similarity_score", ascending=False)

    return ranked.reset_index(drop=True)

def build_method_comparison_table() -> pd.DataFrame:
    rows = [
        {
            "method": "Bag-of-Words",
            "advantages": "Simple, fast, interpretable term frequencies.",
            "limitations": "Ignores context and word order; favors frequent terms.",
            "suitable_applications": "Baselines, sparse linear models, quick diagnostics.",
        },
        {
            "method": "TF-IDF",
            "advantages": "Highlights informative terms; down-weights common words.",
            "limitations": "Still ignores semantics and long-range context.",
            "suitable_applications": "Search, ranking, topic-centric relevance scoring.",
        },
        {
            "method": "Word2Vec",
            "advantages": "Captures semantic similarity and neighborhood structure.",
            "limitations": "Needs sufficient data and tuning; less directly interpretable.",
            "suitable_applications": "Semantic exploration, similarity expansion, query enrichment.",
        },
    ]
    return pd.DataFrame(rows)

def recommend_pipeline() -> str:
    return (
        "Recommended pipeline: TF-IDF for document-level relevance ranking, "
        "supplemented by Word2Vec for semantic term exploration. "
        "Use Bag-of-Words as a baseline for interpretability checks."
    )

data_loader_mod.COMMON_TEXT_COLUMN_NAMES = COMMON_TEXT_COLUMN_NAMES
data_loader_mod._score_text_column = _score_text_column
data_loader_mod._detect_text_column = _detect_text_column
data_loader_mod.load_tweets_dataframe = load_tweets_dataframe

preprocessing_mod.URL_RE = URL_RE
preprocessing_mod.HASHTAG_RE = HASHTAG_RE
preprocessing_mod.MENTION_RE = MENTION_RE
preprocessing_mod.REPEATED_CHARS_RE = REPEATED_CHARS_RE
preprocessing_mod.EMOJI_RE = EMOJI_RE
preprocessing_mod.TOKEN_RE = TOKEN_RE
preprocessing_mod.NoiseStats = NoiseStats
preprocessing_mod._ensure_nltk_resources = _ensure_nltk_resources
preprocessing_mod.compute_noise_stats = compute_noise_stats
preprocessing_mod.preprocess_text_series = preprocess_text_series

representations_mod.build_bow = build_bow
representations_mod.build_tfidf = build_tfidf
representations_mod.corpus_vocabulary_size = corpus_vocabulary_size
representations_mod.top_terms_from_bow = top_terms_from_bow
representations_mod.top_terms_from_tfidf = top_terms_from_tfidf
representations_mod.train_word2vec = train_word2vec
representations_mod.most_similar_terms = most_similar_terms

similarity_mod.rank_similarity_to_reference = rank_similarity_to_reference

evaluation_mod.build_method_comparison_table = build_method_comparison_table
evaluation_mod.recommend_pipeline = recommend_pipeline

sys.modules["src.data_loader"] = data_loader_mod
sys.modules["src.preprocessing"] = preprocessing_mod
sys.modules["src.representations"] = representations_mod
sys.modules["src.similarity"] = similarity_mod
sys.modules["src.evaluation"] = evaluation_mod

print("Embedded modules registered: src.data_loader, src.preprocessing, src.representations, src.similarity, src.evaluation")

## Dataset Setup (after embedded modules)
This cell runs after the embedded module registration so `load_tweets_dataframe` resolves from in-notebook modules and works standalone in Colab.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

from src.data_loader import load_tweets_dataframe

KAGGLE_DATASET_HANDLE = os.getenv("KAGGLE_DATASET_HANDLE", "gpreda/covid19_tweets")
IN_COLAB = "google.colab" in sys.modules
raw_dir = Path("/content/data_raw") if IN_COLAB else (Path.cwd() / "data" / "raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Prefer loader-driven download when available; fallback supports the embedded loader signature.
try:
    _df_probe, detected_text_col, selected_csv = load_tweets_dataframe(
        raw_dir=raw_dir,
        minimum_rows=1,
        sample_size=0,
        random_state=42,
        auto_download=True,
        dataset_handle=KAGGLE_DATASET_HANDLE,
    )
except TypeError:
    import kagglehub

    download_root = Path(kagglehub.dataset_download(KAGGLE_DATASET_HANDLE))
    csv_files = list(download_root.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in downloaded dataset path: {download_root}")

    for csv_path in csv_files:
        target = raw_dir / csv_path.name
        shutil.copy2(csv_path, target)

    _df_probe, detected_text_col, selected_csv = load_tweets_dataframe(
        raw_dir=raw_dir,
        minimum_rows=1,
        sample_size=0,
        random_state=42,
    )

print(f"Dataset ready via load_tweets_dataframe from: {selected_csv}")
print(f"Detected text column in downloaded data: {detected_text_col}")
print(f"CSV files available in raw dir: {raw_dir}")
for f in sorted(raw_dir.glob("*.csv")):
    print(" -", f.name)

# COVID-19 Public Discussion Analysis

This notebook implements a complete NLP pipeline on the Kaggle COVID-19 tweets dataset:
- Text preprocessing and normalization
- Feature representation with Bag-of-Words, TF-IDF, and Word2Vec
- Document similarity analysis with cosine similarity
- Evaluation and recommendation of representation methods

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_tweets_dataframe
from src.preprocessing import compute_noise_stats, preprocess_text_series
from src.representations import (
    build_bow,
    build_tfidf,
    corpus_vocabulary_size,
    most_similar_terms,
    top_terms_from_bow,
    top_terms_from_tfidf,
    train_word2vec,
    )
from src.similarity import rank_similarity_to_reference
from src.evaluation import build_method_comparison_table, recommend_pipeline

RANDOM_STATE = 42
MIN_REQUIRED_ROWS = 1000
SAMPLE_SIZE = int(os.getenv("PIPELINE_SAMPLE_SIZE", "10000"))
REFERENCE_TOPIC = "covid pandemic vaccine lockdown coronavirus public health"
QUERY_TERMS = ["covid", "vaccine", "lockdown", "pandemic", "coronavirus"]

if IN_COLAB and Path("/content/data_raw").exists():
    RAW_DIR = Path("/content/data_raw")
else:
    RAW_DIR = PROJECT_ROOT / "data" / "raw"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

## 1) Load Dataset (Using Pandas, minimum 1000 tweets)

In [ ]:
df, text_col, source_path = load_tweets_dataframe(
    raw_dir=RAW_DIR,
    minimum_rows=MIN_REQUIRED_ROWS,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Source file: {source_path}")
print(f"Detected text column: {text_col}")
print(f"Rows analyzed: {len(df)}")

## 2) Display 10 Example Tweets

In [ ]:
examples = df[text_col].head(10).reset_index(drop=True)
pd.DataFrame({"tweet_example": examples})

## 3) Basic Dataset Statistics

In [ ]:
tweet_lengths_chars = df[text_col].str.len()
raw_tokens = df[text_col].str.lower().str.split()
raw_vocab = {tok for row in raw_tokens for tok in row}

stats_df = pd.DataFrame([
    {
        "number_of_tweets": len(df),
        "avg_tweet_length_chars": float(tweet_lengths_chars.mean()),
        "vocabulary_size_raw": len(raw_vocab),
    }
])
stats_df

## 4) Noise and Formatting Inspection
Identify URLs, hashtags, mentions, emojis, and repeated characters.

In [ ]:
noise = compute_noise_stats(df[text_col])
noise_df = pd.DataFrame([noise.__dict__])
noise_df

## 5) Text Cleaning Pipeline
Lowercase, remove URLs/punctuation/special chars, tokenize, remove stopwords, lemmatize.

In [ ]:
cleaned = preprocess_text_series(df[text_col])
df = pd.concat([df.reset_index(drop=True), cleaned], axis=1)

cleaned_out = PROCESSED_DIR / "covid_tweets_cleaned.csv"
df.to_csv(cleaned_out, index=False)
print(f"Saved cleaned dataset to: {cleaned_out}")

df[[text_col, "clean_text", "clean_tokens"]].head(5)

## 6) Bag-of-Words (CountVectorizer)

In [ ]:
corpus = [str(x) for x in df["clean_text"].tolist()]
tokenized_corpus = [list(x) for x in df["clean_tokens"].tolist()]

bow_vectorizer, bow_matrix = build_bow(corpus=corpus, max_features=20000, min_df=2)
bow_shape = bow_matrix.shape
bow_vocab_size = len(bow_vectorizer.get_feature_names_out())
bow_top_terms = top_terms_from_bow(bow_vectorizer, bow_matrix, top_k=20)

print(f"BoW matrix shape: {bow_shape}")
print(f"BoW vocabulary size: {bow_vocab_size}")
pd.DataFrame(bow_top_terms, columns=["term", "count"]).head(20)

### BoW Interpretation
Check whether meaningful COVID terms appear among top terms and whether generic frequent words dominate.

In [ ]:
covid_terms = {"covid", "coronavirus", "pandemic", "vaccine", "lockdown", "health"}
top_bow_set = {t for t, _ in bow_top_terms}
present = sorted(covid_terms.intersection(top_bow_set))
missing = sorted(covid_terms.difference(top_bow_set))
print("COVID terms found in BoW top terms:", present)
print("COVID terms not in BoW top terms:", missing)

## 7) TF-IDF (TfidfVectorizer)

In [ ]:
tfidf_vectorizer, tfidf_matrix = build_tfidf(corpus=corpus, max_features=20000, min_df=2)
tfidf_shape = tfidf_matrix.shape
tfidf_vocab_size = len(tfidf_vectorizer.get_feature_names_out())
tfidf_top_terms = top_terms_from_tfidf(tfidf_vectorizer, tfidf_matrix, top_k=20)

print(f"TF-IDF matrix shape: {tfidf_shape}")
print(f"TF-IDF vocabulary size: {tfidf_vocab_size}")
pd.DataFrame(tfidf_top_terms, columns=["term", "mean_tfidf_score"]).head(20)

### TF-IDF Interpretation
Observe how highly frequent words are down-weighted relative to distinctive discussion terms.

In [ ]:
top_tfidf_set = {t for t, _ in tfidf_top_terms}
present_tfidf = sorted(covid_terms.intersection(top_tfidf_set))
print("COVID terms found in TF-IDF top terms:", present_tfidf)

## 8) Word2Vec Embeddings (gensim)

In [ ]:
w2v_model = train_word2vec(
    tokenized_corpus=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=5,
    workers=max(1, (os.cpu_count() or 2) - 1),
    epochs=8,
    seed=RANDOM_STATE,
)

print("Word2Vec vocabulary size:", len(w2v_model.wv))
print("Cleaned vocabulary size:", corpus_vocabulary_size(tokenized_corpus))

similar_dict = most_similar_terms(w2v_model, QUERY_TERMS, topn=10)
rows = []
for query, items in similar_dict.items():
    if not items:
        rows.append({"query_term": query, "similar_word": None, "score": None})
    else:
        for word, score in items:
            rows.append({"query_term": query, "similar_word": word, "score": score})

pd.DataFrame(rows).head(50)

## 9) Cosine Similarity to Reference Topic
Reference text: `covid pandemic vaccine lockdown coronavirus public health`

In [ ]:
ranked = rank_similarity_to_reference(
    tweets=df["clean_text"],
    reference_text=REFERENCE_TOPIC,
    max_features=20000,
    min_df=2,
)

top_related = ranked.head(10).copy()
least_related = ranked.tail(10).copy()

top_related

In [ ]:
least_related

## 10) Evaluation of Representation Methods

In [ ]:
comparison_df = build_method_comparison_table()
comparison_df

In [ ]:
print("Recommended representation pipeline:")
print(recommend_pipeline())

## 11) Save Notebook Outputs
Export key artifacts for report writing and reproducibility.

In [ ]:
comparison_df.to_csv(OUTPUTS_DIR / "representation_comparison.csv", index=False)
ranked.to_csv(OUTPUTS_DIR / "tweet_similarity_ranking.csv", index=False)
w2v_model.save(str(OUTPUTS_DIR / "word2vec_covid.model"))

print("Saved:")
print(OUTPUTS_DIR / "representation_comparison.csv")
print(OUTPUTS_DIR / "tweet_similarity_ranking.csv")
print(OUTPUTS_DIR / "word2vec_covid.model")

## 11b) Save Outputs to Google Drive (Optional)
Mount Google Drive and copy saved artifacts into a folder in your Drive.

In [ ]:
import shutil
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive

    # Mount Drive (re-running is safe).
    drive.mount("/content/drive", force_remount=False)

    # Destination inside your Google Drive.
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/coursework_2/outputs")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    artifacts = [
        OUTPUTS_DIR / "representation_comparison.csv",
        OUTPUTS_DIR / "tweet_similarity_ranking.csv",
        OUTPUTS_DIR / "word2vec_covid.model",
    ]

    for artifact in artifacts:
        if not artifact.exists():
            raise FileNotFoundError(f"Missing artifact: {artifact}. Run the save-outputs cell first.")
        target = DRIVE_OUTPUT_DIR / artifact.name
        shutil.copy2(artifact, target)
        print(f"Saved to Google Drive: {target}")
else:
    print("Skipping Google Drive export: not running in Colab.")

## 12) Final Discussion Template
Use this section in your report:
- Advantages, limitations, and suitable applications of BoW
- Advantages, limitations, and suitable applications of TF-IDF
- Advantages, limitations, and suitable applications of Word2Vec
- Final selection of the most suitable representation pipeline for this COVID discussion task